# QuashKV Demo

This notebook demonstrates the core capabilities of QuashKV — a near-optimal KV cache compression library based on the [TurboQuant paper](https://arxiv.org/abs/2504.19874) (ICLR 2026).

**Sections:**
1. Lloyd-Max Codebook
2. MSE & Inner Product Quantizers
3. KV Cache Engine
4. Mixed Precision (Outlier Handling)
5. Vector Similarity Search
6. HuggingFace Cache Integration
7. Benchmarking

In [ ]:
import torch
import numpy as np
import time

# QuashKV imports
from quashkv import (
    LloydMaxCodebook,
    MSEQuantizer,
    InnerProductQuantizer,
    QuashKVEngine,
    QuashIndex,
    MixedPrecisionMSEQuantizer,
    MixedPrecisionIPQuantizer,
    pack_bits,
    unpack_bits,
)
from quashkv.codebook import compute_mse_cost
from quashkv.integrations.hf_cache import QuashKVCache

torch.manual_seed(42)
print(f"PyTorch {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Lloyd-Max Codebook

The foundation of QuashKV is the Lloyd-Max optimal scalar quantizer. After random rotation,
each coordinate follows a known distribution (Beta → Gaussian), and we precompute the
optimal codebook centroids and decision boundaries.

In [ ]:
# Build codebooks for different bit-widths
for bits in [2, 3, 4]:
    cb = LloydMaxCodebook(bits=bits, dim=128)
    n_levels = 2 ** bits
    
    print(f"\n--- {bits}-bit codebook (d=128), {n_levels} levels ---")
    print(f"  Centroids: {cb.centroids.numpy().round(4)}")
    print(f"  Boundaries: {cb.boundaries.numpy().round(4)}")
    
    # Theoretical MSE cost
    mse = compute_mse_cost(bits, dim=128)
    print(f"  Theoretical MSE cost: {mse:.6f}")

In [ ]:
# Verify quantization round-trip
cb = LloydMaxCodebook(bits=3, dim=128)
x = torch.randn(1000)  # standard normal (post-rotation distribution)

indices = cb.quantize(x)
x_hat = cb.dequantize(indices)

mse = ((x - x_hat) ** 2).mean().item()
print(f"Empirical MSE: {mse:.6f}")
print(f"Theoretical:   {compute_mse_cost(3, 128):.6f}")
print(f"Indices range: [{indices.min().item()}, {indices.max().item()}]")

## 2. MSE & Inner Product Quantizers

- **MSEQuantizer**: Random rotation Π → per-coordinate Lloyd-Max → rescale. Minimizes MSE.
- **InnerProductQuantizer**: (b-1)-bit MSE + 1-bit QJL for unbiased inner product estimation.

In [ ]:
d = 64
n_vectors = 500

# Generate test data on unit sphere
X = torch.randn(n_vectors, d)
X = X / X.norm(dim=-1, keepdim=True)

# MSE Quantizer
mse_q = MSEQuantizer(d=d, bits=3, seed=42)
indices, norms = mse_q.compress(X)
X_hat = mse_q.decompress(indices, norms)

mse_error = ((X - X_hat) ** 2).mean().item()
cos_sim = torch.nn.functional.cosine_similarity(X, X_hat, dim=-1).mean().item()

print("=== MSE Quantizer (3-bit) ===")
print(f"  MSE: {mse_error:.6f}")
print(f"  Cosine similarity: {cos_sim:.4f}")
print(f"  index dtype: {indices.dtype}, shape: {indices.shape}")

In [ ]:
# Inner Product Quantizer — preserves dot products
ip_q = InnerProductQuantizer(d=d, bits=3, seed=42)

ip_indices, ip_norms = ip_q.compress(X)
X_ip_hat = ip_q.decompress(ip_indices, ip_norms)

# Check IP preservation
query = torch.randn(10, d)
query = query / query.norm(dim=-1, keepdim=True)

true_ips = query @ X[:20].T       # (10, 20)
approx_ips = query @ X_ip_hat[:20].T

ip_error = ((true_ips - approx_ips) ** 2).mean().item()
ip_corr = torch.corrcoef(torch.stack([true_ips.flatten(), approx_ips.flatten()]))[0, 1].item()

print("=== Inner Product Quantizer (3-bit) ===")
print(f"  IP MSE: {ip_error:.6f}")
print(f"  IP correlation: {ip_corr:.4f}")
print(f"  Unbiased: E[<q, x_hat>] ≈ <q, x>")

## 3. KV Cache Engine

The `QuashKVEngine` orchestrates compression and attention over the full KV cache.
It uses MSEQuantizer for values and InnerProductQuantizer for keys.

In [ ]:
head_dim = 128
n_heads = 8
batch_size = 1
seq_len = 256

engine = QuashKVEngine(head_dim=head_dim, key_bits=3, value_bits=3, seed=42)

# Simulate prefill: append KV pairs
keys = torch.randn(batch_size, n_heads, seq_len, head_dim)
values = torch.randn(batch_size, n_heads, seq_len, head_dim)

engine.append(keys, values)

print(f"Stored {len(engine.blocks)} compressed blocks")
print(f"Compression ratio: {engine.compression_ratio():.2f}x")
print(f"Compressed memory: {engine.actual_memory_bytes() / 1024:.1f} KB")
print(f"Original memory: {keys.numel() * 4 * 2 / 1024:.1f} KB")

In [ ]:
# Attend over compressed cache
queries = torch.randn(batch_size, n_heads, 1, head_dim)
output = engine.attention(queries)

print(f"Output shape: {output.shape}")
print(f"Output norm: {output.norm(dim=-1).mean().item():.4f}")

# Compare with full-precision attention
scale = head_dim ** -0.5
scores = (queries * scale) @ keys.transpose(-2, -1)
weights = torch.softmax(scores, dim=-1)
ref_output = weights @ values

attn_mse = ((output - ref_output) ** 2).mean().item()
print(f"Attention MSE vs full precision: {attn_mse:.6f}")

In [ ]:
# Enable packed storage for maximum compression
engine_packed = QuashKVEngine(head_dim=head_dim, key_bits=3, value_bits=3, seed=42)
engine_packed.append(keys, values)
engine_packed.pack_storage()

print(f"Unpacked memory: {engine.actual_memory_bytes() / 1024:.1f} KB")
print(f"Packed memory:   {engine_packed.actual_memory_bytes() / 1024:.1f} KB")
print(f"Packed ratio:    {engine_packed.compression_ratio():.2f}x")

# Attention still works after packing
output_packed = engine_packed.attention(queries)
pack_diff = ((output - output_packed) ** 2).mean().item()
print(f"Packed vs unpacked attention diff: {pack_diff:.2e}")

## 4. Mixed Precision (Outlier Handling)

Some KV channels have much higher variance (outliers). Mixed precision gives these
channels more bits while keeping regular channels at lower precision.

In [ ]:
d = 128
n_cal = 200

# Create data with outlier channels (channels 0-7 have 10x variance)
cal_data = torch.randn(n_cal, d)
cal_data[:, :8] *= 10.0  # Simulate outlier channels

# Standard quantizer
std_q = MSEQuantizer(d=d, bits=2, seed=42)
std_idx, std_norms = std_q.compress(cal_data)
std_recon = std_q.decompress(std_idx, std_norms)
std_mse = ((cal_data - std_recon) ** 2).mean().item()

# Mixed precision: outlier channels get 3 bits, regular get 2 bits
mp_q = MixedPrecisionMSEQuantizer(
    d=d, outlier_bits=3, regular_bits=2, n_outliers=16, seed=42,
)
mp_q.calibrate(cal_data)

mp_idx, mp_norms = mp_q.compress(cal_data)
mp_recon = mp_q.decompress(mp_idx, mp_norms)
mp_mse = ((cal_data - mp_recon) ** 2).mean().item()

print(f"Standard 2-bit MSE:       {std_mse:.4f}")
print(f"Mixed 2/3-bit MSE:        {mp_mse:.4f}")
print(f"Improvement:              {std_mse / mp_mse:.2f}x")
print(f"Effective bits:           {mp_q.effective_bits():.2f}")
print(f"Detected outlier channels: {sorted(mp_q.outlier_mask.nonzero().squeeze(-1).tolist())[:10]}...")

## 5. Vector Similarity Search

`QuashIndex` provides approximate maximum inner-product search (MIPS) using
the InnerProductQuantizer — with zero training time.

In [ ]:
d = 64
N = 5000
n_queries = 50

# Database vectors
db = torch.randn(N, d)
db = db / db.norm(dim=-1, keepdim=True)

# Build index — no training!
t0 = time.perf_counter()
index = QuashIndex(dim=d, bits=3, seed=42)
index.add(db)
idx_time = time.perf_counter() - t0

# Queries
queries = torch.randn(n_queries, d)
queries = queries / queries.norm(dim=-1, keepdim=True)

# Search
t0 = time.perf_counter()
scores, top_indices = index.search(queries, k=10)
search_time = time.perf_counter() - t0

# Recall
recall = index.recall_at_k(queries, db, k=10)

print(f"Database: {N} vectors, d={d}")
print(f"Index time: {idx_time*1000:.1f} ms")
print(f"Search time: {search_time*1000:.1f} ms ({n_queries} queries)")
print(f"Recall@10: {recall:.3f}")
print(f"Compression ratio: {index.compression_ratio():.1f}x")
print(f"Memory: {index.memory_bytes() / 1024:.1f} KB (vs {N * d * 4 / 1024:.1f} KB float32)")

In [ ]:
# Recall vs bit-width
print("\nRecall@10 by bit-width:")
print(f"{'Bits':>6} {'Recall@10':>10} {'Ratio':>8}")
print("-" * 28)

for bits in [2, 3, 4]:
    idx = QuashIndex(dim=d, bits=bits, seed=42)
    idx.add(db)
    r = idx.recall_at_k(queries, db, k=10)
    cr = idx.compression_ratio()
    print(f"{bits:>6} {r:>10.3f} {cr:>7.1f}x")

## 6. HuggingFace Cache Integration

`QuashKVCache` is a drop-in replacement for HuggingFace's `DynamicCache`,
compressing KV pairs on-the-fly during generation.

In [ ]:
num_layers = 4
head_dim = 128

cache = QuashKVCache(num_layers=num_layers, head_dim=head_dim, key_bits=3, value_bits=3)

# Simulate multi-layer generation
for step in range(8):
    for layer_idx in range(num_layers):
        k = torch.randn(1, 8, 16, head_dim)  # 16 new tokens per step
        v = torch.randn(1, 8, 16, head_dim)
        cache.update(k, v, layer_idx)

stats = cache.compression_stats()
print(f"Layers: {num_layers}")
print(f"Steps: 8 × 16 = 128 tokens per layer")
print(f"Total compressed memory: {stats['total_compressed_bytes'] / 1024:.1f} KB")
print(f"Total original memory: {stats['total_original_bytes'] / 1024:.1f} KB")
print(f"Overall compression: {stats['overall_ratio']:.2f}x")

# Cache is iterable
print(f"\nCache length: {len(cache)} layers")
for i, (keys, values) in enumerate(cache):
    if i == 0:
        print(f"Layer 0 compressed — {len(cache[0][0].blocks)} blocks")

## 7. Bit Packing

Quantized indices are packed into dense byte arrays for minimum memory footprint.

In [ ]:
# Demonstrate packing for each bit-width
d = 128
print(f"{'Bits':>5} {'Indices':>10} {'Packed':>10} {'Ratio':>8} {'Lossless':>10}")
print("-" * 48)

for bits in [1, 2, 3, 4]:
    indices = torch.randint(0, 2**bits, (1000, d), dtype=torch.uint8)
    packed = pack_bits(indices, bits)
    unpacked = unpack_bits(packed, bits, d)
    
    lossless = (indices == unpacked).all().item()
    ratio = indices.numel() / packed.numel()
    
    print(f"{bits:>5} {indices.numel():>10,} {packed.numel():>10,} {ratio:>7.1f}x {'✓' if lossless else '✗':>10}")

## 8. Benchmarking

Quick micro-benchmarks for key operations.

In [ ]:
head_dim = 128
n_heads = 32
seq_len = 512

# Compression timing
engine = QuashKVEngine(head_dim=head_dim, key_bits=3, value_bits=3)
keys = torch.randn(1, n_heads, seq_len, head_dim)
values = torch.randn(1, n_heads, seq_len, head_dim)

times = []
for _ in range(5):
    engine.blocks.clear()
    t0 = time.perf_counter()
    engine.append(keys, values)
    times.append(time.perf_counter() - t0)
compress_ms = np.median(times) * 1000

# Attention timing
queries = torch.randn(1, n_heads, 1, head_dim)
times = []
for _ in range(5):
    t0 = time.perf_counter()
    _ = engine.attention(queries)
    times.append(time.perf_counter() - t0)
attn_ms = np.median(times) * 1000

print(f"Config: {n_heads} heads, d={head_dim}, seq_len={seq_len}")
print(f"Compression: {compress_ms:.1f} ms")
print(f"Attention:   {attn_ms:.1f} ms")
print(f"Cache size:  {engine.actual_memory_bytes() / 1024:.1f} KB")
print(f"Compression: {engine.compression_ratio():.2f}x")

In [ ]:
# Scaling: compression ratio vs sequence length
print(f"\n{'Seq Len':>10} {'Ratio':>8} {'Compress (ms)':>15} {'Attn (ms)':>12}")
print("-" * 50)

for sl in [128, 256, 512, 1024, 2048]:
    e = QuashKVEngine(head_dim=128, key_bits=3, value_bits=3)
    k = torch.randn(1, 8, sl, 128)
    v = torch.randn(1, 8, sl, 128)
    
    t0 = time.perf_counter()
    e.append(k, v)
    c_t = (time.perf_counter() - t0) * 1000
    
    q = torch.randn(1, 8, 1, 128)
    t0 = time.perf_counter()
    _ = e.attention(q)
    a_t = (time.perf_counter() - t0) * 1000
    
    print(f"{sl:>10,} {e.compression_ratio():>7.2f}x {c_t:>14.1f} {a_t:>11.1f}")

---

**Next steps:**
- Run `benchmarks/quality_bench.py` for full MSE/cosine analysis
- Run `benchmarks/latency_bench.py` for comprehensive timing
- Try `benchmarks/nn_search_bench.py --large` for 100K-vector recall curves
- Install Triton on GPU for fused kernel acceleration